<center><img src="https://github.com/pandas-dev/pandas/raw/main/web/pandas/static/img/pandas.svg" alt="pandas Logo" style="width: 800px;"/></center>

# Merging Pandas DataFrames

This notebook will combine into one dataframe observations from two separate data sources: the NYS Mesonet, and ASOS sites located in or near New York State. 

## Overview

Often, we wish to analyze and visualize datasets from multiple sources. The datasets may seem similar, but they may be formatted differently; they may share some common variables but differ in others; and/or they may use different physical units. 

In this notebook, we will read in two separate sources of surface meteorological observations. One is from the New York State Mesonet; the other is from a worldwide network of surface observation sites primarily located at airports. These latter observations follow an international standard, known as METAR. While some METAR sites are staffed by people who add additional information to the observations, they all report observations using **Automated Surface Observing Systems** (ASOS).

Once we read in an example set of hourly observations from both sources, we will work towards generating a merged Pandas DataFrame. We will need to standardize different column names, and convert units so they are all in Metric.

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| Pandas | Necessary | |
| MetPy | Necessary | |
| Datetime | Necessary | |

**Time to learn**: 30 minutes

## Imports

In [ ]:
import pandas as pd
import numpy as np
import metpy.calc as mpcalc
from metpy.units import units

## Read in and examine the NYSM and ASOS-derived datafiles for a particular hour

Also read in the NYSM site table so we can extract latitude, longitude, and elevation. 

In [ ]:
nysm_data_file = '/spare11/atm533/data/nysm_2025090320.csv'
nysm_sites_file = '/spare11/atm533/data/nysm_sites.csv'
asos_data_file = '/spare11/atm533/data/asos_2025090320.csv'

In [ ]:
nysm_data = pd.read_csv(nysm_data_file)
nysm_sites = pd.read_csv(nysm_sites_file)

Examine the ASOS data, using a Jupyterlab feature that allows us to execute Linux commands, as if we were typing them in the terminal. Note the following differences in this dataset's format as compared to how the NYSM data is structured:
1. Its columns are named differently from those of the NYSM
1. Its columns are separated using whitespace, not tabs
1. Missing values are denoted by -9999.0, not null strings
1. The date/time is formatted differently.

### Execute a Linux command by prefacing the next code line with an `!`, and enclosing the Python variable that corresponds to the file we will open.

In [ ]:
! head $asos_data_file

### Open the file with Pandas; as part of the call to `read_csv`, specify that columns are separated by one or more blank spaces, unlike the default (comma-separated values). Treat any cell with certain values as missing.

In [ ]:
asos_data = pd.read_csv(asos_data_file, sep='\\s+',na_values=['-9999.0', '-9999.00'])

Look at the first couple of rows in all three DataFrames.

In [ ]:
nysm_data.head(2)

In [ ]:
nysm_sites.head(2)

In [ ]:
asos_data.head(2)

#### Remove (*drop*) unwanted columns from a DataFrame

Our merged data file will contain only a subset of the columns from each data source. First, let's remove the unwanted columns from the NYSM file.

Use `inplace` so the Dataframe object is updated

In [ ]:
nysm_data.drop(columns=['temp_9m [degC]','precip_incremental [mm]', 'precip_local [mm]', 'precip_max_intensity [mm/min]',
       'avg_wind_speed_prop [m/s]', 'max_wind_speed_prop [m/s]','wind_speed_stddev_prop [m/s]', 'wind_direction_prop [degrees]',
       'wind_direction_stddev_prop [degrees]','wind_speed_stddev_sonic [m/s]','wind_direction_stddev_sonic [degrees]', 'solar_insolation [W/m^2]','snow_depth [cm]', 'frozen_soil_05cm [bit]',
       'frozen_soil_25cm [bit]', 'frozen_soil_50cm [bit]',
       'soil_temp_05cm [degC]', 'soil_temp_25cm [degC]',
       'soil_temp_50cm [degC]', 'soil_moisture_05cm [m^3/m^3]',
       'soil_moisture_25cm [m^3/m^3]', 'soil_moisture_50cm [m^3/m^3]'],inplace=True)

Examine what columns remain post-drop.

In [ ]:
nysm_data.columns

Now, drop any unwanted columns from the ASOS data file. The file has 35 columns, so rather than creating a long list of column names to drop, let's create a list of only the columns we wish to keep.

In [ ]:
asos_keep = ['STN', 'YYMMDD/HHMM', 'SLAT', 'SLON', 'SELV', 'TMPC', 'DWPC', 'PMSL', 'ALTI', 'SKNT', 'GUST', 'DRCT']

In [ ]:
asos_data.columns

Redefine the ASOS dataframe, which will now contain only the desired columns.

In [ ]:
asos_data = asos_data[asos_keep]

Examine the columns remaining in the ASOS data file

In [ ]:
asos_data.columns

#### These columns represent the following:
1. Station ID
2. Date and Time
3. Latitude
4. Longitude
5. Elevation
6. 2-meter temperature
7. 2-meter dewpoint
8. Sea-level pressure in hPa (not all stations report this)
9. Altimeter setting in inches of mercury (not all stations report this)
10. Wind speed in knots
11. Peak wind gust in knots
12. Wind direction in degrees

#### Use a Python *dictionary* to rename columns

Each dataframe has varying column names. Let's standardize by creating a `dictionary` that will map current column names to common (and in some cases, much shorter) names.

In [ ]:
column_mapping = {'station' : 'STID',
                  'time': 'TIME',
                  'temp_2m [degC]': 'TMPC',
                  'relative_humidity [percent]': 'RELH',
                  'precip_incremental [mm]': 'PRCP',
                  'precip_local [mm]': 'P24M',
                  'avg_wind_speed_sonic [m/s]': 'SPED',
                  'max_wind_speed_sonic [m/s]': 'GUMS',
                  'wind_direction_sonic [degrees]': 'DRCT', 
                  'station_pressure [mbar]': 'PRES',
                  'stid': 'STID',
                  'name': 'NAME',
                  'lat': 'SLAT',
                  'lon': 'SLON',
                  'elevation': 'SELV',
                  'STN':  'STID',
                  'YYMMDD/HHMM': 'TIME'}

For each of the three Dataframes, rename the columns according to our dictionary. Then examine each Dataframe to see how they look.

In [ ]:
nysm_data.rename(columns=column_mapping, inplace=True)

In [ ]:
nysm_data.head()

In [ ]:
nysm_sites.rename(columns=column_mapping, inplace=True)

In [ ]:
nysm_sites.head()

In [ ]:
asos_data.rename(columns=column_mapping, inplace=True)

In [ ]:
asos_data.head()

Parse the time strings in the NYSM and ASOS dataframes into their corresponding `datetime` representations.

In [ ]:
nysm_data['TIME'] = pd.to_datetime(nysm_data['TIME'],format="%Y-%m-%d %H:%M:%S UTC", utc=True)
asos_data['TIME'] = pd.to_datetime(asos_data['TIME'],format="%y%m%d/%H%M", utc=True)

### For the ASOS data, convert non-Metric variables to Metric and then remove the non-metric columns in the `DataFrame`

In [ ]:
asos_data['SPED'] = (asos_data['SKNT'] * units('kts').to('m/s'))
asos_data['GUMS'] = (asos_data['GUST'] * units('kts').to('m/s'))
asos_data['ALTM'] = (asos_data['ALTI'] * units('inHg').to('hPa'))

asos_data.drop(columns=['SKNT', 'GUST', 'ALTI'], inplace=True)

Examine the ASOS data again

In [ ]:
asos_data

#### Merge columns from one DataFrame into another, and then re-index.</span>

Use `merge` to add the latitude, longitude, and elevation from the NYSM Sites dataframe to the NYSM Data one.
To do this, we take a subset of the nysm_sites dataframe containing the ones we want to merge, plus the station id column, since `merge` requires a common column for both dataframes. Note also that `merge` does not have an `inplace` option so we just redefine the `nysm_data` object.

In [ ]:
nysm_data = pd.merge(nysm_data,nysm_sites[['STID','SLAT','SLON','SELV']])

In [ ]:
nysm_data.head()

#### Make the NYSM and ASOS dataframes *multi-indexed* according to *site* and *time*. Convert the date/time from string to Datetime objects. 

First set the indexes

In [ ]:
nysm_data.set_index(['STID', 'TIME'], inplace = True)
asos_data.set_index(['STID', 'TIME'], inplace = True)

In [ ]:
nysm_data.head()

In [ ]:
asos_data.head()

#### Add columns to a DataFrame

The NYSM data does not contain dewpoint or sea-level pressure yet. Calculate and create columns for dewpoint and sea-level pressure.

In [ ]:
# Reduce station pressure to SLP. Source: https://www.sandhurstweather.org.uk/barometric.pdf 
sensorHeight = .5
nysm_data['PMSL'] = nysm_data['PRES'] * np.exp((nysm_data['SELV'] + sensorHeight)/((nysm_data['TMPC'] + 273.15) * 29.263))

We'll use MetPy to calculate dewpoint from temperature and relative humidity. Since MetPy's calculation library typically requires units to be attached to variables used in the calculations, do so next.

In [ ]:
tmpc = nysm_data['TMPC'].values * units ('degC')
rh = nysm_data['RELH'].values * units('percent')

nysm_data['DWPC'] = mpcalc.dewpoint_from_relative_humidity(tmpc, rh)

Now that we have calculated sea-level pressure and dewpoint, let's drop station pressure and relative humidity from the NYSM DataFrame since they do not appear in the ASOS one.

In [ ]:
nysm_data.drop(columns=['PRES','RELH'],inplace=True,axis='columns')

In [ ]:
nysm_data

### Concatenate one DataFrame into another

Now, the two dataframes can be merged. We'll use the `concat` method to "glue" the rows from the ASOS table on to the NYSM one.

In [ ]:
nymerge_data = pd.concat([nysm_data,asos_data])

In [ ]:
nymerge_data

Let's rearrange the columns into a more logical order, and eliminate any that are not common to both of the original NYSM and ASOS dataframes.

In [ ]:
colOrder = ['SLAT','SLON','SELV','TMPC','DWPC','PMSL','DRCT','SPED','GUMS']

In [ ]:
nymerge_data = nymerge_data.reindex(columns=colOrder)

<div class="alert alert-block alert-warning">
<b>Note:</b> Whenever you combine DataFrames, it's a very good idea to re-sort the indices. Although the new DataFrame may "look" correct, its indices may not be in what Pandas calls <b>lexical order</b>. We'll use Pandas <code>sort_index</code> for this purpose.</div>

In [ ]:
nymerge_data.sort_index(ascending=True, inplace=True)

In [ ]:
nymerge_data

### Analyze the merged DataFrame

Now, we have what we wanted ... one Dataframe that contains the same variables ... in the same units ... for both datasets. We can make selections on this multi-index Dataframe.

Get some general information about the `DataFrame`:

In [ ]:
nymerge_data.info()

Select one station

In [ ]:
nymerge_data.loc['VOOR']

Select multiple stations and one time

In [ ]:
nymerge_data.loc[(('ALB','VOOR'),('2025-09-03 20:00:00')),('TMPC','DWPC')]

Use the *cross-section* method to select one index value and then display all values of the other index for selected columns.

In [ ]:
nymerge_data.xs('2025-09-03 20:00:00 UTC',level='TIME')[['TMPC','DWPC','SPED']]

<div class="alert alert-block alert-warning">
    <b>Note:</b> As currently written, this notebook reads in data for just a single time. Since we have made <b>TIME</b> a Pandas <code>Index</code>instead of a mere column, selections based on multiple times or even a time range could easily be done.</div>

<div class="alert alert-block alert-info">
    <b>Next step:</b> We could, of course, take this merged <code>DataFrame</code> and create a station plot with it ... which would show not only the NYSM data, but also the traditional ASOS network of stations!</div>

## Summary
* Pandas can efficiently combine datasets with different attributes into a single `DataFrame`.
* A Pandas `DataFrame` can have, and benefit from, multiple indexes.

## Resources and References
1. [MetPy Monday Episode 94](https://www.youtube.com/watch?v=ncpYohRYG3I&list=PLQut5OXpV-0ir4IdllSt1iEZKTwFBa7kO&index=85&t=4s)
1. [MetPy Monday Episode 97](https://www.youtube.com/watch?v=rj2ZEAIbg1k&list=PLQut5OXpV-0ir4IdllSt1iEZKTwFBa7kO&index=88&ab_channel=Unidata)
1. [MetPy Monday Episode 98](https://www.youtube.com/watch?v=slUGaLyLJX0&list=PLQut5OXpV-0ir4IdllSt1iEZKTwFBa7kO&index=89&ab_channel=Unidata)
1. [National Weather Service Automated Surface Observing Systems (ASOS)](https://www.weather.gov/asos/)